# FinSight AI model training

This notebook rebuilds the two artifacts consumed by `app/ml_service.py`. It addresses the original model's missing reproducibility and evaluation by using explicit targets, train/test splits, preprocessing pipelines, fixed feature order, and regression/classification metrics.

Important: the source dataset does not contain an `ai_quality_tag` target. The quality label below is a documented proxy derived from the existing `rating` field: ratings 4-5 are `Good`, ratings 2-3 are `Average`, and ratings 0-1 are `Risky`. Replace this rule with reviewed labels if you have a better ground-truth definition.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'fastapi_funds_dataset.csv'
MODEL_PATH = ROOT / 'models'
RANDOM_STATE = 42
TEST_SIZE = 0.20
print(f'Project root: {ROOT}')
print(f'Dataset: {DATA_PATH}')

In [ ]:
df = pd.read_csv(DATA_PATH)

REG_FEATURES = [
    'sharpe', 'alpha', 'beta', 'sortino',
    'fund_size_cr', 'min_sip', 'min_lumpsum'
]
CLF_FEATURES = [
    'sharpe', 'alpha', 'beta', 'sortino',
    'fund_size_cr', 'expense_ratio', 'returns_1yr',
    'risk_level', 'fund_age_yr'
]
REG_TARGET = 'returns_3yr'
REQUIRED = REG_FEATURES + CLF_FEATURES + [REG_TARGET, 'rating']
missing_columns = sorted(set(REQUIRED) - set(df.columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

numeric_columns = sorted(set(REG_FEATURES + CLF_FEATURES + [REG_TARGET, 'rating']))
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df = df.dropna(subset=[REG_TARGET, 'rating']).reset_index(drop=True)
print(f'Rows used: {len(df)}')
print(df[REG_FEATURES + CLF_FEATURES + [REG_TARGET, 'rating']].isna().sum())

## Targets and preprocessing

The return target is the observed 3-year annualized return. The classifier target is a proxy quality label based only on the dataset's rating. Numeric imputation is fitted inside each pipeline, preventing information from the test set leaking into training.

In [ ]:
def quality_from_rating(rating):
    if rating >= 4:
        return 'Good'
    if rating <= 1:
        return 'Risky'
    return 'Average'

y_reg = df[REG_TARGET].astype(float)
y_quality = df['rating'].map(quality_from_rating)

reg_train, reg_test = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, random_state=RANDOM_STATE
)
clf_train, clf_test = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=y_quality
)

def numeric_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', model),
    ])

regressor = numeric_pipeline(RandomForestRegressor(
    n_estimators=300, max_depth=10, min_samples_leaf=2,
    random_state=RANDOM_STATE, n_jobs=-1
))
classifier = numeric_pipeline(RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=2,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
))

regressor.fit(df.loc[reg_train, REG_FEATURES], y_reg.iloc[reg_train])
classifier.fit(df.loc[clf_train, CLF_FEATURES], y_quality.iloc[clf_train])

reg_predictions = regressor.predict(df.loc[reg_test, REG_FEATURES])
clf_predictions = classifier.predict(df.loc[clf_test, CLF_FEATURES])
print('Quality distribution:')
print(y_quality.value_counts())

In [ ]:
rmse = mean_squared_error(y_reg.iloc[reg_test], reg_predictions) ** 0.5
print('Regression evaluation')
print(f'MAE: {mean_absolute_error(y_reg.iloc[reg_test], reg_predictions):.3f} percentage points')
print(f'RMSE: {rmse:.3f} percentage points')
print(f'R2: {r2_score(y_reg.iloc[reg_test], reg_predictions):.3f}')

print('Classification evaluation')
print(f'Accuracy: {accuracy_score(y_quality.iloc[clf_test], clf_predictions):.3f}')
print(classification_report(y_quality.iloc[clf_test], clf_predictions, zero_division=0))
print('Confusion matrix labels:', classifier.named_steps['model'].classes_)
print(confusion_matrix(y_quality.iloc[clf_test], clf_predictions, labels=classifier.named_steps['model'].classes_))

## Save compatible artifacts

The saved pipelines accept the same pandas column names used by `app/ml_service.py`. The label encoder is retained for the current service contract. Do not replace production artifacts until the evaluation results and sample API responses have been reviewed.

In [ ]:
MODEL_PATH.mkdir(parents=True, exist_ok=True)

quality_encoder = LabelEncoder()
quality_encoder.fit(['Average', 'Good', 'Risky'])

joblib.dump(regressor, MODEL_PATH / 'returns_3yr_predictor.joblib')
joblib.dump(classifier, MODEL_PATH / 'fund_classifier.joblib')
joblib.dump(quality_encoder, MODEL_PATH / 'fund_label_encoder.joblib')

metadata = {
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'regression_target': REG_TARGET,
    'regression_features': REG_FEATURES,
    'classification_features': CLF_FEATURES,
    'quality_target_rule': 'rating >= 4: Good; rating <= 1: Risky; otherwise: Average',
    'regression_mae': round(float(mean_absolute_error(y_reg.iloc[reg_test], reg_predictions)), 4),
    'regression_rmse': round(float(rmse), 4),
    'regression_r2': round(float(r2_score(y_reg.iloc[reg_test], reg_predictions)), 4),
    'classification_accuracy': round(float(accuracy_score(y_quality.iloc[clf_test], clf_predictions)), 4),
}
(MODEL_PATH / 'training_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved models and training_metadata.json')

## Next improvements

- Collect time-based fund histories and use a chronological split for realistic forecasting.
- Train separate horizon models or use horizon as an explicit feature instead of applying a 3-year prediction to 1-15 year plans.
- Replace rating-derived quality labels with analyst-reviewed or outcome-based labels.
- Add prediction intervals, fees, taxes, and downside scenarios to portfolio projections.
- Add a risk constraint to plan selection so individual funds match the selected profile.